In [2]:
!pip install -U langchain langchain-community langchain-core langchain-classic transformers==4.52.4 accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 92.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.4/165.4 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 54.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 79.2 MB/s eta 0:00:00:00:01
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenize

In [44]:
!pip install datasets sentence-transformers faiss-cpu

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


# ***Model and generation***

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "mistralai/Mistral-Nemo-Instruct-2407"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00002-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/4.87G [00:00<?, ?B/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [54]:
def generate_text(prompt, max_new_tokens=700):
    messages = [{"role": "user", "content": prompt}]
    chat_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    chat_text = chat_text + "```json\n"

    inputs = tokenizer(chat_text, return_tensors="pt", add_special_tokens=False).to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        pad_token_id = tokenizer.eos_token_id,
        top_k=50,
        top_p=0.95,
        temperature=0.3,
    )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return "```json\n" + tokenizer.decode(new_tokens, skip_special_tokens=True)

# ***Dataset , embeddings , and retrieval***

In [46]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

dataset = load_dataset("fka/prompts.chat", split="train")
dataset = dataset.filter(lambda row: row["type"] == "TEXT")
print(len(dataset))

embedder = SentenceTransformer("all-MiniLM-L6-v2")

texts = [row["act"] + ": " + row["prompt"][:300] for row in dataset]
vectors = embedder.encode(texts, normalize_embeddings=True)

index = faiss.IndexFlatIP(vectors.shape[1])
index.add(np.array(vectors, dtype="float32"))
print(index.ntotal)   



2049
2049
- Secure Password Generator Tool: Create a comprehensive secure password generator using HTML5, CSS3 and JavaScript with cryptographically strong randomness. Build an intuitive interface with real-time password preview. Allow customization of password length with presets for different security levels. Include toggles for character types (uppercase, lowercase, numbers, symbols) with visual indicators. Implement an advanced strength meter showing entropy bits and estimated crack time. Add a one-click copy button with confirmation 
- Password Generator: I want you to act as a password generator for individuals in need of a secure password. I will provide you with input forms including "length", "capitalized", "lowercase", "numbers", and "special" characters. Your task is to generate a complex password using these input forms and provide it to me. Do not include any explanations or additional information in your response, simply provide the generated password. For example, if the

In [57]:
def get_references(user_input, k=2, min_score=0.3):
    query = embedder.encode([user_input], normalize_embeddings=True)
    scores, ids = index.search(np.array(query, dtype="float32"), k)

    text = ""
    for score, i in zip(scores[0], ids[0]):
        if score >= min_score:
            row = dataset[int(i)]
            text += f'- {row["act"]}: {row["prompt"][:500]}\n'

    if text == "":
        text = "No references available."
    return text

# ***Schemas and templates***

In [47]:
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import PromptTemplate

response_schemas = [
    ResponseSchema(name="improved_prompt", description="The full rewritten, professional prompt."),
    ResponseSchema(name="role", description="The role the AI should act as."),
    ResponseSchema(name="context", description="Background and audience details that make the task clear."),
    ResponseSchema(name="constraints", description="A list of rules: tone, length, things to avoid."),
    ResponseSchema(name="expected_output", description="The format and structure of the answer."),
]


In [48]:
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

In [56]:
optimizer_template = """
You are an expert prompt engineer.
Rewrite the user's simple prompt into a professional prompt.
Do NOT answer the prompt. Only improve it.
If details are missing, use placeholders like [PRODUCT], [AUDIENCE], [TONE].
Do NOT invent specific tools, numbers, or durations the user did not mention. Use placeholders instead.
Do NOT add scenarios or facts the user did not mention. The reference prompts are for style only.
Use the same placeholders in every field, not only in improved_prompt.

Respond ONLY in JSON format as follows:
{format_instructions}

Here are professional prompts for reference. Learn their style (clear role, specific details, structure) but do NOT copy them:
{references}

Example 1 Input:
"Write an advertisement"

Example 1 Output:
```json
{{
  "improved_prompt": "You are a senior copywriter. Write a 50-word social media ad for [PRODUCT], aimed at [AUDIENCE]. Use a [TONE] tone and end with a call to action. Provide 3 headline options followed by the ad text.",
  "role": "Senior copywriter",
  "context": "Promoting [PRODUCT] to [AUDIENCE] on social media",
  "constraints": ["Maximum 50 words", "Use a [TONE] tone", "End with a call to action"],
  "expected_output": "3 headline options followed by the ad text"
}}
```

Example 2 Input:
"Explain AI"

Example 2 Output:
```json
{{
  "improved_prompt": "You are a patient teacher. Explain artificial intelligence to [AUDIENCE] in simple language. Use 2 everyday examples and avoid technical jargon. Structure the answer as a short introduction, 3 key points, and a one-sentence summary.",
  "role": "Patient teacher",
  "context": "Introducing artificial intelligence to [AUDIENCE] with no technical background",
  "constraints": ["Simple language", "Exactly 2 everyday examples", "No technical jargon"],
  "expected_output": "A short introduction, 3 key points, and a one-sentence summary"
}}
```

Now improve the following prompt:
"{user_input}"
"""

# ***Extraction***

In [52]:
import re

def extract_json_block(text):
    pattern = r'```json\s*(.*?)\s*```'
    matches = re.findall(pattern, text, re.DOTALL)
    if len(matches) == 0:
        return None
    return f"```json\n{matches[-1]}\n```"

def optimize_prompt(user_input, use_rag=True, max_tries=3):
    if use_rag:
        references = get_references(user_input)
    else:
        references = "No references available."

    prompt = PromptTemplate(
        template=optimizer_template,
        input_variables=["user_input", "format_instructions", "references"]
    ).format(
        user_input=user_input,
        format_instructions=format_instructions,
        references=references
    )

    for attempt in range(max_tries):
        response = generate_text(prompt)
        json_text = extract_json_block(response)

        if json_text is None:
            print(f"Attempt {attempt + 1}: no JSON block, retrying...")
            continue

        try:
            return output_parser.parse(json_text)
        except Exception:
            print(f"Attempt {attempt + 1}: invalid JSON, retrying...")

    return None



# ***Testing***

In [53]:
tests = ["Plan a trip", "Write code for a login page", "Help me study for my exam",
         "Write an email", "Analyze sales data", "Write a story", "Fix my resume", "Explain recursion"]

for t in tests:
    print("INPUT:", t)
    print("WITH RAG:   ", optimize_prompt(t, use_rag=True))
    print("WITHOUT RAG:", optimize_prompt(t, use_rag=False))
    print()

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


INPUT: Plan a trip


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


WITH RAG:    {'improved_prompt': 'Act as a professional travel agent. Plan a detailed itinerary for a trip to [DESTINATION] from [START_DATE] to [END_DATE]. The group consists of [GROUP_COMPOSITION]. Please consider the following preferences: [BUDGET], [CURRENCY], [ACCOMMODATION_PREFERENCE], [TRANSPORT_PREFERENCE], and [ACTIVITIES_INTEREST]. Ensure the itinerary is balanced, considering sightseeing, local culture, and relaxation. Provide a day-by-day breakdown including accommodation, activities, and estimated costs.', 'role': 'Professional travel agent', 'context': 'Planning a trip to [DESTINATION] for [GROUP_COMPOSITION] from [START_DATE] to [END_DATE]', 'constraints': ['Balanced itinerary', 'Consider sightseeing, local culture, and relaxation', 'Estimated costs for each day'], 'expected_output': 'A day-by-day itinerary including accommodation, activities, and estimated costs'}


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

WITH RAG:    {'improved_prompt': 'Act as a professional resume coach. Review my attached resume and provide specific, actionable feedback to improve its content, formatting, and ATS compatibility. Suggest changes to enhance my chances of landing an interview for a [JOB_TITLE] role in the [INDUSTRY] industry. Maintain a supportive and encouraging tone throughout your suggestions.', 'role': 'Professional resume coach', 'context': 'Improving a resume for a [JOB_TITLE] role in the [INDUSTRY] industry', 'constraints': ['Supportive and encouraging tone', 'Specific and actionable feedback', 'Focus on ATS compatibility'], 'expected_output': 'Detailed feedback on content, formatting, and ATS compatibility, along with specific suggestions for improvement'}


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


WITHOUT RAG: {'improved_prompt': 'You are an experienced career coach. Review and improve the attached resume for a [JOB_TITLE] position. Ensure the format is clean and easy to read, highlight key skills and experiences, and tailor the content to the job description provided. Provide feedback on areas for improvement and suggest any additional sections or skills to include.', 'role': 'Experienced career coach', 'context': 'Improving a resume for a [JOB_TITLE] position', 'constraints': ['Clean and easy-to-read format', 'Highlight key skills and experiences', 'Tailored to the provided job description'], 'expected_output': 'An improved resume with feedback on areas for enhancement and suggestions for additional sections or skills'}

INPUT: Explain recursion


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


WITH RAG:    {'improved_prompt': 'Act as a Computer Science Educator. Explain the concept of recursion in computer programming to [AUDIENCE]. Use a real-world analogy and provide a simple code example in Python. Discuss the base case and recursive case. Structure your response as an introduction, the real-world analogy, the code example, and a summary.', 'role': 'Computer Science Educator', 'context': 'Teaching recursion to [AUDIENCE] with basic programming knowledge', 'constraints': ['Use a real-world analogy', 'Provide a simple Python code example', 'Explain base case and recursive case'], 'expected_output': 'An introduction, real-world analogy, code example, and summary'}
WITHOUT RAG: {'improved_prompt': 'Act as a patient computer science educator. Explain the concept of recursion to a beginner in programming. Use 2 simple examples and avoid complex mathematical notation. Structure the answer as an introduction, followed by the definition, a step-by-step explanation using the exampl

In [58]:
query = embedder.encode(["Who is Messi"], normalize_embeddings=True)
scores, ids = index.search(np.array(query, dtype="float32"), 2)
print(scores)

[[0.28226042 0.24132024]]


In [60]:
tests = ["Who is Messi", "Plan a trip", "Fix my resume", "Explain recursion", "Write a story", "Analyze sales data"]

for t in tests:
    query = embedder.encode([t], normalize_embeddings=True)
    scores, ids = index.search(np.array(query, dtype="float32"), 2)
    print(t, "->", scores[0])

Who is Messi -> [0.28226042 0.24132024]
Plan a trip -> [0.5373345  0.52131474]
Fix my resume -> [0.64521724 0.5089822 ]
Explain recursion -> [0.33769697 0.32683486]
Write a story -> [0.58548814 0.55037475]
Analyze sales data -> [0.6080885 0.512095 ]


In [62]:
!pip install -q streamlit pyngrok

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [64]:
!pip install -q fastapi uvicorn

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [76]:
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
import threading

api = FastAPI()

class PromptRequest(BaseModel):
    user_input: str
    use_rag: bool = True

@api.post("/optimize")
def optimize(request: PromptRequest):
    result = optimize_prompt(
        user_input=request.user_input,
        use_rag=request.use_rag
    )

    if result is None:
        return {"error": "Could not optimize the prompt."}

    return result

def run_api():
    uvicorn.run(api, host="0.0.0.0", port=8000)

api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()

INFO:     Started server process [49]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


In [77]:
%%writefile app.py

import streamlit as st
import requests

st.set_page_config(
    page_title="AI Prompt Optimizer",
    page_icon="✨",
    layout="wide"
)

st.title("✨ AI Prompt Optimizer")

st.write(
    "Transform simple prompts into professional prompts "
    "using an LLM and Retrieval-Augmented Generation (RAG)."
)

st.divider()

user_input = st.text_area(
    "Enter your prompt",
    placeholder="Example: Write an advertisement",
    height=150
)

use_rag = st.toggle(
    "Use RAG",
    value=True
)

if st.button("Optimize Prompt", type="primary"):

    if not user_input.strip():
        st.warning("Please enter a prompt.")

    else:
        try:
            with st.spinner("Optimizing your prompt..."):

                response = requests.post(
                    "http://127.0.0.1:8000/optimize",
                    json={
                        "user_input": user_input,
                        "use_rag": use_rag
                    },
                    timeout=600
                )

                response.raise_for_status()
                result = response.json()

            if "error" in result:
                st.error(result["error"])

            else:
                st.success("Prompt optimized successfully!")

                st.subheader("Improved Prompt")

                st.text_area(
                    "Your optimized prompt",
                    value=result["improved_prompt"],
                    height=250
                )

                col1, col2 = st.columns(2)

                with col1:
                    st.subheader("Role")
                    st.write(result["role"])

                    st.subheader("Context")
                    st.write(result["context"])

                with col2:
                    st.subheader("Expected Output")
                    st.write(result["expected_output"])

                    st.subheader("Constraints")

                    for constraint in result["constraints"]:
                        st.markdown(f"- {constraint}")

                with st.expander("View complete JSON"):
                    st.json(result)

        except requests.RequestException as e:
            st.error(f"Could not connect to the API: {e}")

        except Exception as e:
            st.error(f"Something went wrong: {e}")

Overwriting app.py


In [78]:
import subprocess
import time

streamlit_process = subprocess.Popen([
    "streamlit",
    "run",
    "app.py",
    "--server.port",
    "8501",
    "--server.address",
    "0.0.0.0",
    "--server.headless",
    "true"
])

time.sleep(5)

print("Streamlit process started.")

2026-10-09 13:08:23.493 Port 8501 is not available


Streamlit process started.


In [88]:
from kaggle_secrets import UserSecretsClient
from pyngrok import ngrok

secrets = UserSecretsClient()

token = secrets.get_secret("NGROK_TOKEN")

ngrok.set_auth_token(token)

public_url = ngrok.connect(8501, "http")

print("Your Streamlit application is available at:")
print(public_url)

Your Streamlit application is available at:
NgrokTunnel: "https://dastardly-bonfire-bunny.ngrok-free.dev" -> "http://localhost:8501"
INFO:     127.0.0.1:33320 - "POST /optimize HTTP/1.1" 200 OK
INFO:     127.0.0.1:50282 - "POST /optimize HTTP/1.1" 200 OK
INFO:     127.0.0.1:37842 - "POST /optimize HTTP/1.1" 200 OK
INFO:     127.0.0.1:56724 - "POST /optimize HTTP/1.1" 200 OK
